In [9]:
%load_ext autoreload
%autoreload 2
from scripts.utils import *
from scripts.gaussianclassifiers import *
from scripts.dcf import *
from scripts.logisticreg import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 📘 Theory & Key Functions
**Theoretical Recall:** Logistic Regression is a discriminative model that learns a linear decision boundary by directly optimizing the posterior probability via the logistic sigmoid function. It minimizes a regularized cross-entropy loss using L-BFGS. 
**Quadratic Expansion** maps the input features into a higher-dimensional space (adding $x_i x_j$ interaction terms), allowing the implicitly linear solver to learn a non-linear decision boundary in the original feature space.

**Key Functions (from `logisticreg.py`):**
- `trainLogReg(DTR, LTR, l)`: Defines the cross-entropy objective and its analytical gradient, passing them to `scipy.optimize.fmin_l_bfgs_b` to learn weights $w$ and bias $b$.
- `weighted_trainLogReg(DTR, LTR, l, piT)`: A variant that artificially balances the classes during training by weighting errors according to a target prior $\pi_T$.
- `expand_features(D)`: Applies the quadratic transformation $\phi(x) = [\text{vec}(xx^T); x]$ to all samples.

See the implementations in [logisticreg.py](../scripts/logisticreg.py).

In [10]:
D,L,labels=init_dataset()

In [11]:
(DTR, LTR), (DVAL, LVAL)=split_db_2to1(D,L)

### Training the Linear Logistic Regression
We define the regularized objective function minimized by the L-BFGS solver using the analytical gradient, which improves speed and stability compared to the approximated numerical gradient.

In [6]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 2.377797e-01
Error Rate: 9.3%
DCF min: 0.36397529441884274
Actual DCF:0.9440924219150024

Lambda: 0.00031622776601683794
J ottima: 2.391006e-01
Error Rate: 9.3%
DCF min: 0.3649673579109063
Actual DCF:0.9531650025601638

Lambda: 0.001
J ottima: 2.430980e-01
Error Rate: 9.3%
DCF min: 0.3649673579109063
Actual DCF:0.9531650025601638

Lambda: 0.0031622776601683794
J ottima: 2.543260e-01
Error Rate: 9.2%
DCF min: 0.36411930363543266
Actual DCF:0.9350198412698413

Lambda: 0.01
J ottima: 2.815594e-01
Error Rate: 9.2%
DCF min: 0.3611431131592422
Actual DCF:0.9350198412698413

Lambda: 0.03162277660168379
J ottima: 3.353240e-01
Error Rate: 9.2%
DCF min: 0.3621351766513057
Actual DCF:0.9340277777777778

Lambda: 0.1
J ottima: 4.198415e-01
Error Rate: 9.2%
DCF min: 0.36411930363543266
Actual DCF:0.9431003584229389

Lambda: 0.31622776601683794
J ottima: 5.223593e-01
Error Rate: 9.3%
DCF min: 0.36397529441884274
Actual DCF:0.9521729390681003

Lambda: 1.0
J ottima: 6.107602e-0

### Experiment 1: Linear model on the full training set
In this phase we test different values of $\lambda$ on the entire training set.

Metric interpretation:
- **Error Rate**: classification accuracy with a fixed threshold
- **minDCF**: best possible cost by varying the threshold
- **actualDCF**: cost at the chosen operating point (with prior $\pi_T$ and costs)

If `actualDCF` is much higher than `minDCF`, the model discriminates but the scores are not well calibrated for that operating point.

In [7]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR[:,::50],LTR[::50],l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 1.197749e-01
Error Rate: 11.1%
DCF min: 0.4466045826932924
Actual DCF:1.5050403225806452

Lambda: 0.00031622776601683794
J ottima: 1.255871e-01
Error Rate: 10.9%
DCF min: 0.4446204557091654
Actual DCF:1.4778225806451613

Lambda: 0.001
J ottima: 1.386344e-01
Error Rate: 10.8%
DCF min: 0.4487327188940092
Actual DCF:1.4435163850486432

Lambda: 0.0031622776601683794
J ottima: 1.636818e-01
Error Rate: 10.5%
DCF min: 0.44972478238607266
Actual DCF:1.3729198668714797

Lambda: 0.01
J ottima: 2.070173e-01
Error Rate: 10.4%
DCF min: 0.44065220174091135
Actual DCF:1.3466941884280594

Lambda: 0.03162277660168379
J ottima: 2.759238e-01
Error Rate: 9.8%
DCF min: 0.4147145417306707
Actual DCF:1.2216621863799282

Lambda: 0.1
J ottima: 3.738010e-01
Error Rate: 9.8%
DCF min: 0.39884152585765487
Actual DCF:1.1388728878648233

Lambda: 0.31622776601683794
J ottima: 4.895101e-01
Error Rate: 9.2%
DCF min: 0.3887768817204301
Actual DCF:0.8865367383512546

Lambda: 1.0
J ottima: 5.90643

### Experiment 2: Reduced training (1 sample every 50)
Here we repeat the same procedure with very little training data to observe the effect of the low-sample regime.

Theoretical expectation:
- increase in parameter estimation variance
- greater sensitivity to regularization
- possible worsening of stability and decision cost on validation.

### Prior-weighted Logistic Regression
This variant introduces different weights for positive/negative samples according to the target prior $\pi_T$.

The goal is to align the training to the application operating point, penalizing more heavily the errors on the most critical class.

In [8]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=weighted_trainLogReg(DTR,LTR,l,pi_T)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-target_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(llr, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.0001
J ottima: 1.295710e-01
Error Rate: 16.9%
DCF min: 0.37205581157194056
Actual DCF:0.407050051203277

Lambda: 0.00031622776601683794
J ottima: 1.307709e-01
Error Rate: 17.0%
DCF min: 0.3700716845878136
Actual DCF:0.4009536610343062

Lambda: 0.001
J ottima: 1.343060e-01
Error Rate: 17.5%
DCF min: 0.36992767537122373
Actual DCF:0.4128584229390681

Lambda: 0.0031622776601683794
J ottima: 1.436651e-01
Error Rate: 18.6%
DCF min: 0.36695148489503326
Actual DCF:0.43269969278033793

Lambda: 0.01
J ottima: 1.642912e-01
Error Rate: 21.8%
DCF min: 0.36298323092677925
Actual DCF:0.4487007168458782

Lambda: 0.03162277660168379
J ottima: 2.002131e-01
Error Rate: 29.6%
DCF min: 0.36397529441884274
Actual DCF:0.5963741679467486

Lambda: 0.1
J ottima: 2.473025e-01
Error Rate: 46.1%
DCF min: 0.36482334869431643
Actual DCF:0.9146825396825398

Lambda: 0.31622776601683794
J ottima: 2.889006e-01
Error Rate: 50.4%
DCF min: 0.36397529441884274
Actual DCF:1.0

Lambda: 1.0
J ottima: 3.117064e-01
Er

### Quadratic Feature Expansion
To increase the expressive capacity of the linear model, we expand each sample with quadratic terms $xx^T$ in addition to the linear terms.

In this way, the decision remains linear in the expanded space but becomes non-linear in the original space.

In [9]:
def expand_features(D):
    n_features = D.shape[0]
    n_samples = D.shape[1]
    
    # La nuova dimensionalità sarà data dal quadrato delle feature originali (xx^T)
    # sommato al numero delle feature originali (x)
    expanded_D = np.zeros((n_features**2 + n_features, n_samples))
    
    for i in range(n_samples):
        # Estrazione del singolo campione come vettore colonna
        x = D[:, i:i+1]
        
        # Calcolo della matrice dei prodotti incrociati
        xxT = np.dot(x, x.T)
        
        # Vettorizzazione della matrice e concatenazione con il termine lineare
        phi_x = np.vstack([xxT.reshape(-1, 1), x])
        
        # Inserimento del campione espanso nella nuova matrice
        expanded_D[:, i:i+1] = phi_x
        
    return expanded_D

# Esempio di utilizzo prima di addestrare il modello:
DTR_quad = expand_features(DTR)
DVAL_quad = expand_features(DVAL)

### Experiment 3: Quadratic Logistic Regression
Finally we evaluate the model after feature expansion, comparing the metrics with the linear version.

In [15]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR_quad,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL_quad)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 1.513656e-01
Error Rate: 6.0%
DCF min: 0.2602246543778802
Actual DCF:0.5887096774193548

Lambda: 0.00031622776601683794
J ottima: 1.534955e-01
Error Rate: 6.0%
DCF min: 0.2612167178699437
Actual DCF:0.5887096774193548

Lambda: 0.001
J ottima: 1.596177e-01
Error Rate: 5.9%
DCF min: 0.25865655401945725
Actual DCF:0.57864503328213

Lambda: 0.0031622776601683794
J ottima: 1.751964e-01
Error Rate: 5.9%
DCF min: 0.2527041730670763
Actual DCF:0.5614919354838709

Lambda: 0.01
J ottima: 2.086068e-01
Error Rate: 5.9%
DCF min: 0.2487359190988223
Actual DCF:0.5453309011776754

Lambda: 0.03162277660168379
J ottima: 2.686447e-01
Error Rate: 5.9%
DCF min: 0.243631592421915
Actual DCF:0.5049283154121864

Lambda: 0.1
J ottima: 3.594710e-01
Error Rate: 6.0%
DCF min: 0.24660778289810548
Actual DCF:0.5079045058883769

Lambda: 0.31622776601683794
J ottima: 4.716574e-01
Error Rate: 6.1%
DCF min: 0.2629128264208909
Actual DCF:0.5412186379928315

Lambda: 1.0
J ottima: 5.762647e-01
Err

Let's save the best result obtained, we will need it in subsequent notebooks.

In [10]:
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))
l=0.03162277660168379
v_opt,J_min=trainLogReg(DTR_quad,LTR,l)
w_opt=v_opt[:-1]
b_opt=v_opt[-1]
S_val=np.dot(w_opt.T,DVAL_quad)+b_opt
LP=(S_val>0).astype(int)
error_rate=np.mean(LP!=LVAL)
score_lr_quad=S_val-target_prior_log_odds
np.save('llr/score_lr_best.npy', score_lr_quad)